In [1]:
import pandas as pd
import numpy as np
import os

## Crop Data Processing Pipeline

This notebook processes agricultural crop data from Excel files containing information about various crops across different districts. The data includes metrics like area under cultivation, production, and yield across different seasons (Kharif, Rabi, and Whole Year).

### Data Structure

The original data files have a horizontal crop layout with vertical year progression, containing:

- **Location**: State and District information
- **Time Period**: Year ranges (e.g., 2012-2013)
- **Crops**: Multiple crops including Rice, Wheat, Sugarcane, pulses, oilseeds, etc.
- **Seasons**: Kharif (monsoon), Rabi (winter), and Whole Year
- **Metrics**: Area (Hectares), Production (Tonnes/Bales), and Yield (Tonne/Hectare)

### Processing Steps

The pipeline performs the following transformations:

1. **Data Loading**: Reads Excel files from the `data/xls_files` directory
2. **Column Flattening**: Converts MultiIndex columns into a flat structure with format `Crop_Season_Metric`
3. **Data Transformation**: Reshapes wide format data into long format with columns:
   - State, District, Season, Year
   - Crop, Area_Ha, Production_Ton, Yield_TonHa
4. **Data Cleaning**:
   - Removes numbering prefixes from state/district names
   - Handles missing values and empty strings
   - Drops rows where all three metrics are missing
5. **Output**: Appends processed data to `data/converted_crop_data.csv`


In [2]:
def process_df(
    df: pd.DataFrame, output_file="data/converted_crop_data.csv"
) -> pd.DataFrame:
    # Convert the data to the required format
    output_rows = []

    # Get column names and organize them by crop and season
    # Each crop-season combination has 3 columns: Area, Production, Yield
    columns = df.columns.tolist()

    # Find all crop-season combinations
    crop_season_cols = {}
    for col in columns[3:]:  # Skip State, District, Year
        if "_" in col:
            parts = col.split("_")
            if len(parts) >= 3:
                crop = parts[0]
                season = parts[1]
                metric = parts[2]  # 'Area', 'Production', or 'Yield'

                key = (crop, season)
                if key not in crop_season_cols:
                    crop_season_cols[key] = {}

                if "Area" in metric:
                    crop_season_cols[key]["Area"] = col
                elif "Yield" in metric:
                    crop_season_cols[key]["Yield"] = col

    # Process data rows (skip first 3 header rows)
    for idx in range(3, len(df)):
        row = df.iloc[idx]
        state = row.iloc[0] if pd.notna(row.iloc[0]) else ""
        district = row.iloc[1] if pd.notna(row.iloc[1]) else ""
        year = row.iloc[2] if pd.notna(row.iloc[2]) else ""

        # Skip if no district or year
        if not district or not year:
            continue

        # Clean state/district name (remove numbering)
        state = state.split(". ")[-1] if ". " in str(state) else state
        district = district.split(". ")[-1] if ". " in str(district) else district

        # Create output rows for each crop-season combination
        for (crop, season), col_names in crop_season_cols.items():
            area = np.nan
            yield_val = np.nan

            # Get values from respective columns
            if "Area" in col_names:
                val = row[col_names["Area"]]
                if pd.notna(val) and str(val).strip() != "":  # type: ignore
                    try:
                        area = float(str(val).replace(",", ""))
                    except:  # noqa: E722
                        area = np.nan

            if "Yield" in col_names:
                val = row[col_names["Yield"]]
                if pd.notna(val) and str(val).strip() != "":  # type: ignore
                    try:
                        yield_val = float(str(val).replace(",", ""))
                    except:  # noqa: E722
                        yield_val = np.nan

            output_rows.append(
                {
                    "State": state,
                    "District": district,
                    "Season": season,
                    "Year": year,
                    "Crop": crop,
                    "Area_Ha": area,
                    "Yield_QHa": yield_val * 10 if pd.notna(yield_val) else np.nan,
                }
            )

    # Create DataFrame from output rows
    output_df = pd.DataFrame(output_rows)

    # Drop rows where all three metrics (Area, Production, Yield) are NaN
    output_df = output_df.dropna(subset=["Area_Ha", "Yield_QHa"], how="all")

    # Reset index after dropping rows
    output_df = output_df.reset_index(drop=True)

    # Save to CSV
    # Check if the file already exists
    if os.path.exists(output_file):
        # Append without header
        output_df.to_csv(output_file, mode="a", index=False, header=False)
        print(f"Appended {len(output_df)} rows to existing file: {output_file}")
    else:
        # Create new file with header
        output_df.to_csv(output_file, index=False)
        print(f"Created new file and saved {len(output_df)} rows to {output_file}")

    print(f"Total rows being written this run: {len(output_df)}")

    return output_df

In [3]:
processed_df = None
for f in os.listdir("data/xls_files"):
    if f.endswith(".xls") or f.endswith(".xlsx"):
        file_path = os.path.join("data/xls_files", f)
        df = pd.read_html(file_path)[0]

        # Flatten the MultiIndex columns
        if isinstance(df.columns, pd.MultiIndex):
            # Flatten by combining the levels: (Crop, Season, Metric)
            df.columns = [
                "_".join(
                    [str(c) for c in col if c not in ["State", "District", "Year"]]
                ).strip("_")
                if any(c not in ["State", "District", "Year"] for c in col)
                else col[0]
                for col in df.columns
            ]

        processed_df = process_df(df)

processed_df

Created new file and saved 12980 rows to data/converted_crop_data.csv
Total rows being written this run: 12980
Appended 6451 rows to existing file: data/converted_crop_data.csv
Total rows being written this run: 6451
Appended 3682 rows to existing file: data/converted_crop_data.csv
Total rows being written this run: 3682
Appended 9267 rows to existing file: data/converted_crop_data.csv
Total rows being written this run: 9267
Appended 7958 rows to existing file: data/converted_crop_data.csv
Total rows being written this run: 7958
Appended 9 rows to existing file: data/converted_crop_data.csv
Total rows being written this run: 9
Appended 75 rows to existing file: data/converted_crop_data.csv
Total rows being written this run: 75
Appended 11 rows to existing file: data/converted_crop_data.csv
Total rows being written this run: 11
Appended 36 rows to existing file: data/converted_crop_data.csv
Total rows being written this run: 36
Appended 276 rows to existing file: data/converted_crop_dat

,State,District,Season,Year,Crop,Area_Ha,Yield_QHa
0,West Bengal,24 paraganas north,Whole Year,2015 - 2016,Coconut,3723.0,130271.3
1,West Bengal,24 paraganas north,Rabi,2015 - 2016,Gram,1027.0,6.1
2,West Bengal,24 paraganas north,Rabi,2015 - 2016,Groundnut,486.0,12.5
3,West Bengal,24 paraganas north,Kharif,2015 - 2016,Jute,51274.0,166.2
4,West Bengal,24 paraganas north,Rabi,2015 - 2016,Khesari,471.0,7.3
...,...,...,...,...,...,...,...
4015,West Bengal,Purulia,Rabi,2022 - 2023,Arhar/Tur,454.0,12.6
4016,West Bengal,Purulia,Rabi,2022 - 2023,Horse-gram,1250.0,5.1
4017,West Bengal,Purulia,Kharif,2022 - 2023,Niger seed,354.0,4.1
4018,West Bengal,Purulia,Kharif,2022 - 2023,Bajra,70.0,4.3
